In [17]:
from datetime import datetime
from flask import Flask, request, jsonify
import os, time, random

app = Flask(__name__)

UPLOAD_FOLDER = "uploads"
os.makedirs(UPLOAD_FOLDER, exist_ok=True)

processing_result = None  
result_ready = False  
processing_status = False  # Cờ để báo server đang xử lý
status_ready = False  # Cờ báo ESP32-CAM dừng gửi ảnh

@app.route('/log', methods=['POST'])
def receive_log():
    log_data = request.json.get("log")
    timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[{timestamp}] 📡 Log từ ESP32-CAM: {log_data}")
    return "Log nhận thành công!", 200

@app.route('/upload', methods=['POST'])
def upload_image():
    global processing_result, result_ready, processing_status, status_ready
    filename = f"{UPLOAD_FOLDER}/image_{time.time()}.jpg"
    with open(filename, "wb") as f:
        f.write(request.data)

    print(f"📸 Ảnh đã lưu: {filename}")
    if not processing_status:  # Chỉ xử lý ảnh mới nếu chưa xử lý xong ảnh trước đó
        processing_status = True  # Đánh dấu server đang xử lý
        time.sleep(1)  
        processing_result = random.choice(["servo1", "servo2", "servo3", "servo4"])
        result_ready = True  
        status_ready = True  # Đánh dấu ESP32-CAM có thể dừng gửi ảnh
        processing_status = False  # Hoàn tất xử lý

    return jsonify({"message": "Ảnh nhận thành công!", "file": filename}), 200

@app.route('/check_status', methods=['GET'])
def check_status():
    global status_ready
    if processing_status:
        return jsonify({"status": "processing"}), 202  
    elif status_ready:
        status_ready = False  # Đặt lại sau khi ESP32-CAM đọc
        return jsonify({"status": "done"}), 200  
    return jsonify({"status": "idle"}), 204  

@app.route('/get_result', methods=['GET'])
def get_result():
    global processing_result, result_ready
    if result_ready:
        response = {"servo": processing_result}
        result_ready = False  # Đặt lại sau khi ESP8266 đọc
        return jsonify(response), 200  
    return jsonify({"message": "Chưa có dữ liệu"}), 202  

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000, debug=True, use_reloader=False)


 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://10.10.1.41:5000
Press CTRL+C to quit
10.10.0.185 - - [19/Mar/2025 14:57:21] "POST /log HTTP/1.1" 200 -
10.10.0.185 - - [19/Mar/2025 14:57:22] "POST /log HTTP/1.1" 200 -


[14:57:21] 📡 Log từ ESP32-CAM: [Detection 0] .
[14:57:22] 📡 Log từ ESP32-CAM: [Detection 0] ✅ ESP32-CAM đã kết nối WiFi!


10.10.0.185 - - [19/Mar/2025 14:57:22] "POST /log HTTP/1.1" 200 -
10.10.0.185 - - [19/Mar/2025 14:57:22] "POST /log HTTP/1.1" 200 -


[14:57:22] 📡 Log từ ESP32-CAM: [Detection 0] ✅ Camera đã khởi động thành công!
[14:57:22] 📡 Log từ ESP32-CAM: [Detection 0] 📷 Đã chụp ảnh dummy để ổn định camera


...

[15:02:09] 📡 Log từ ESP32-CAM: [Detection 5] 📷 Ảnh chụp thành công
[15:02:09] 📡 Log từ ESP32-CAM: [Detection 5] 🌐 Đang kết nối tới server...
📸 Ảnh đã lưu: uploads/image_1742371329.5042794.jpg


10.10.0.185 - - [19/Mar/2025 15:02:11] "POST /upload HTTP/1.1" 200 -
10.10.0.185 - - [19/Mar/2025 15:02:12] "POST /log HTTP/1.1" 400 -
10.10.0.185 - - [19/Mar/2025 15:02:12] "GET /check_status HTTP/1.1" 200 -
10.10.0.185 - - [19/Mar/2025 15:02:12] "POST /log HTTP/1.1" 200 -
10.10.0.185 - - [19/Mar/2025 15:02:12] "POST /log HTTP/1.1" 200 -


[15:02:12] 📡 Log từ ESP32-CAM: [Detection 5] ✅ Server đã xử lý xong.
[15:02:12] 📡 Log từ ESP32-CAM: [Detection 5] ⏸ Server đã xử lý xong, ảnh mới đã được gửi để xử lý lại.


10.10.0.185 - - [19/Mar/2025 15:02:16] "POST /log HTTP/1.1" 200 -


[15:02:16] 📡 Log từ ESP32-CAM: [Detection 5] ✅ Kết thúc lần nhận diện.


In [6]:
%tb

SystemExit: 1